In [13]:
pip install pandas scikit-learn emlearn numpy

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import emlearn

# percorsi del dataset 
kaggle_csv_input = "dataset/smart_health_dataset.csv"
target_directory = os.path.dirname(kaggle_csv_input) or "dataset"
header_output_path = os.path.join(target_directory, "vitals_classifier.h")

# ricerca dataset 

print(f"\nFiles nella directory corrente:")
for item in os.listdir('.'):
    full_path = os.path.join('.', item)
    if os.path.isdir(full_path):
        print(f"  📁 {item}/")
        try:
            for sub_item in os.listdir(full_path):
                print(f"     - {sub_item}")
        except:
            print(f"     [non accessibile]")
    else:
        print(f"  📄 {item}")

# Verifichiamo che il file sia presente
if not os.path.exists(kaggle_csv_input):
    print(f"\nFile non trovato: {kaggle_csv_input}")


print(f"\n Trovato CSV in: {kaggle_csv_input}")

# carico dataset 
print("Caricamento del dataset  di Kaggle in corso...")
# Legge il CSV rilevando automaticamente il separatore (, o ;)
dataframe = pd.read_csv(kaggle_csv_input, sep=None, engine='python') 

print(f"Dataset caricato! Trovati {len(dataframe)} campioni reali.")

# Rimuoviamo eventuali righe con valori mancanti (NaN)
dataframe = dataframe.dropna()

# mappo colonne datasetv
target_column = 'Status'
if target_column not in dataframe.columns:
    raise KeyError(f"Target column '{target_column}' non trovato nel CSV")

feature_columns = [col for col in dataframe.columns if col != target_column]
print(f"Feature individuate: {feature_columns}")

# Cconverto feature in valori numerici 
for col in feature_columns + [target_column]:
    dataframe[col] = pd.to_numeric(dataframe[col], errors='coerce')

dataframe_len_before = len(dataframe)
dataframe = dataframe.dropna()
print(f"Rimosse {dataframe_len_before - len(dataframe)} righe non valide.")

X = dataframe[feature_columns].values
y = dataframe[target_column].values

# divisione: 80% Training per l'albero, 20% Test per la validazione
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"-> Campioni usati per l'addestramento: {len(X_train)}")
print(f"-> Campioni usati per il test di validazione: {len(X_test)}")

# addestramento modello Random Forest 
print("\nAddestramento del modello Random Forest sui dati reali di Kaggle...")
# Mantengo 5 alberi e profondità 3 per non saturare la memoria RAM limitata dei nodi in Cooja
random_forest_model = RandomForestClassifier(n_estimators=5, max_depth=3, random_state=42)
random_forest_model.fit(X_train, y_train)

# valutazione accuratezza del modello 
y_pred = random_forest_model.predict(X_test)
print("\n--- REPORT DI ACCURATEZZA TINYML ---")
print(classification_report(y_test, y_pred))
print(f"Gradi di rischio elaborati dal modello: {random_forest_model.classes_}")

# esportazione (emlearn)
print("\nConversione del modello reale in codice C embedded via emlearn...")
c_model = emlearn.convert(random_forest_model, method='inline')
c_model.save(file=header_output_path, name='vitals_rf_model')

print("Modello TinyML pronto e addestrato sull'intero dataset")

[DEBUG] Current working directory: /content
[DEBUG] Percorso cercato: dataset/smart_health_dataset.csv
[DEBUG] Percorso assoluto: /content/dataset/smart_health_dataset.csv

[DEBUG] Files nella directory corrente:
  📁 .config/
     - default_configs.db
     - .last_update_check.json
     - .last_survey_prompt.yaml
     - hidden_gcloud_config_universe_descriptor_data_cache_configs.db
     - active_config
     - .last_opt_in_prompt.yaml
     - configurations
     - logs
     - gce
     - config_sentinel
  📁 dataset/
     - .ipynb_checkpoints
     - vitals_classifier.h
     - smart_health_dataset.csv
  📁 sample_data/
     - README.md
     - anscombe.json
     - mnist_train_small.csv
     - california_housing_test.csv
     - mnist_test.csv
     - california_housing_train.csv

[✓] Trovato CSV in: dataset/smart_health_dataset.csv
Caricamento del dataset reale di Kaggle in corso...
[✓] Dataset caricato! Trovati 5909 campioni reali.
-> Campioni usati per l'addestramento: 4727
-> Campioni usati 